# NetSOP-RAG + AutoGen

A Retrieval-Augmented Generation (RAG) assistant for network operations
runbooks, built as a **multi-agent conversation** with
[AutoGen](https://microsoft.github.io/autogen/) instead of a single
prompt-and-retrieve chain.

## What it does

The notebook answers engineer questions (e.g. *"What's the rollback
procedure if the F5 BIG-IP upgrade fails?"*) using only the content of a
network operations SOP document (`TCS_Network_KB_SOPs.docx` — F5 BIG-IP,
Cisco Nexus, VLAN/trunk, and BGP troubleshooting procedures). Instead of a
single LLM call, the question passes through a small team of AutoGen
agents that each own one responsibility:

- **Retriever_Agent** — turns the question into a search query and calls a
  retrieval tool over the vector store. Does not answer anything itself.
- **Answer_Agent** — answers strictly from the retrieved SOP context:
  exact command syntax, exact step order, pass/fail criteria, and
  rollback steps where relevant. Refuses to guess if the context doesn't
  cover the question.
- **Critic_Agent** — grades that answer against the retrieved context
  (grounded? correct order? citations included?) and only signs off with
  `APPROVED` once every check passes, otherwise sends it back for
  revision.
- **Tool_Executor** — a `UserProxyAgent` that actually executes the
  retrieval tool call and ends the conversation once `Critic_Agent`
  approves.

These four agents sit in a round-robin `GroupChat` managed by a
`GroupChatManager`, so retrieval, drafting, and fact-checking happen as
separate turns instead of being collapsed into one prompt.

## Why this shape

Runbook/SOP content is exactly the kind of text where a normal RAG chain
is risky: a hallucinated command or a reordered upgrade step is not a
stylistic nitpick, it's an outage. Splitting the job across a Retriever,
an Answer writer, and a Critic that must approve before the answer is
returned adds a self-checking step that a single-shot chain doesn't have.
A custom `.docx` loader (`docx_markdown_loader.py`) also preserves tables
as Markdown so that a command and its expected result don't get split
apart during chunking.

## Pipeline

1. Load `TCS_Network_KB_SOPs.docx` and convert it to Markdown (headings
   and tables preserved).
2. Split it into overlapping chunks along heading/paragraph boundaries.
3. Embed the chunks with OpenAI embeddings and store them in a persistent
   Chroma vector store.
4. Wrap similarity search in a `retrieve_context_tool` function that the
   agents can call.
5. Run a question through the `Retriever_Agent -> Answer_Agent ->
   Critic_Agent -> Tool_Executor` group chat and return the approved
   answer.

## Requirements

- Python packages: `langchain-openai`, `langchain-text-splitters`,
  `langchain-chroma`, `python-docx`, `pyautogen`, `python-dotenv`,
  `gradio`
- A `.env` file with `OPENAI_API_KEY`
- The source SOP document, `TCS_Network_KB_SOPs.docx`, in the same folder

---

Part of the `llm-engineering-journey` portfolio -- documenting a hands-on
transition from 16+ years of enterprise network engineering into AI/ML
engineering.


In [19]:
# Imports: LangChain building blocks for embeddings/chunking/vector
# store, the custom .docx-to-markdown loader, the raw OpenAI SDK
# client, dotenv for environment variables, AutoGen for the
# multi-agent framework, and Gradio for a potential UI.

from langchain_openai import OpenAI, OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from docx_markdown_loader import load_docx_as_markdown
from openai import OpenAI
from dotenv import load_dotenv
import os
import autogen
import gradio as gr

In [20]:
# Load environment variables (e.g. OPENAI_API_KEY) from a local
# .env file.

load_dotenv(verbose=True)

True

In [21]:
# Read the OpenAI API key from the environment and print a masked
# prefix, just to confirm it loaded without printing the full key.

openai_api_key= os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API KEY: {openai_api_key[:10]}")

OpenAI API KEY: sk-proj-uC


In [22]:
# Instantiate a raw OpenAI SDK client (kept available for direct
# API calls; the AutoGen agents below use their own llm_config
# instead of this client).

openai_client = OpenAI(api_key=openai_api_key)

In [23]:
# Path to the source SOP document that will be loaded, chunked,
# and embedded into the vector store.

FILEPATH = "TCS_Network_KB_SOPs.docx"
print(f"Filepath: {FILEPATH}")

Filepath: TCS_Network_KB_SOPs.docx


In [24]:
# Convert the .docx SOP document into a single LangChain Document,
# with headings and tables preserved as Markdown (see
# docx_markdown_loader.py) so command/result pairs in tables don't
# get flattened into unstructured text.

raw_doc =load_docx_as_markdown(FILEPATH)
print(f"Loaded {len(raw_doc)} documents with {len(raw_doc[0].page_content)} characters")

Loaded 1 documents with 19245 characters


In [25]:
# Split the SOP document into overlapping chunks along Markdown
# heading/paragraph boundaries, so each chunk stays topically
# coherent and a command stays close to its expected result.

# Split the SOP document into chunks with some overlap
doc_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators= ["\n# ", "\n## ", "\n### ", "\n\n", "\n", " ", ""]
)
document = doc_splitter.split_documents(raw_doc)
print(f"Raw SOP document splitted into {len(document)} documents")
print(f"No of chunks in document : {len(document)}")
print(f"Length of 1st chunk : {len(document[0].page_content)}")
print(f"Metadata : {document[0].metadata}")
print(f"Content : {document[0].page_content}")


Raw SOP document splitted into 31 documents
No of chunks in document : 31
Length of 1st chunk : 549
Metadata : {'source': 'TCS_Network_KB_SOPs.docx'}
Content : NETWORK INFRASTRUCTURE

Knowledge Base — Standard Operating Procedures

Tata Consultancy Services

Network Infrastructure & Security Division

| Document ID | Version | Date | Classification |
| --- | --- | --- | --- |
| TCS-NET-KB-002 | v1.0 | June 2025 | Internal Confidential |

| Sections Covered |
| --- |
| 1. F5 BIG-IP Firmware Upgrade |
| 2. F5 SSL Certificate Renewal |
| 3. Cisco Nexus Switch Firmware Upgrade |
| 4. F5 BIG-IP Failover Testing |
| 5. Switch VLAN Configuration |
| 6. Switch Trunk Configuration |
| 7. BGP Troubleshooting |


In [26]:
# Build (or rebuild) a persistent Chroma vector store from the
# chunks using OpenAI embeddings, and expose it as a retriever for
# semantic search over the SOP content.

db_name = "vector_db_autogen"
openai_embeddings = OpenAIEmbeddings(api_key=openai_api_key)
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function= openai_embeddings).delete_collection()
vector_store = Chroma.from_documents(documents=document, embedding=openai_embeddings, persist_directory=db_name)
vector_count = vector_store._collection.count()
retriever = vector_store.as_retriever(search_kwarg = {"k":4})
print(f"Retrieved {vector_count} vectors")

Retrieved 31 vectors


In [27]:
# AutoGen LLM config list: which model each agent will call
# (GPT-4.1) and the API key to use.

config_list =[{
    "model" : "gpt-4.1",
    "api_key" : openai_api_key,
}]

In [28]:
# Shared LLM config dict (config list + temperature) passed to
# every AutoGen agent defined below.

llm_config = {
    "config_list" : config_list,
    "temperature" : 0.3
}

In [29]:
# Retrieval tool the agents can call: runs a similarity search
# against the vector store and formats the top matching chunks
# (with their source metadata) into a single context string.

def retrieve_context_tool(query: str) -> str:
    """Retrieve the most relevant SOP chunks for a given network operations question."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant SOP content found for this query."
    context_part = []
    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        content = doc.page_content
        context_part.append(f"Source : {source}\n Content : {content}")
    return "\n\n---\n\n".join(context_part)

In [30]:
# Define the four AutoGen agents that make up the pipeline:
# - Retriever_Agent: turns the question into a search query and
#   calls the retrieval tool; never answers the question itself.
# - Answer_Agent: answers strictly from the retrieved SOP context,
#   preserving exact command syntax, step order, pass/fail
#   criteria, and rollback steps, with a section citation.
# - Critic_Agent: grades Answer_Agent's answer against the
#   retrieved context and only replies APPROVED (with the final
#   answer text) once every check passes; otherwise asks for a
#   revision.
# - Tool_Executor: a UserProxyAgent that executes the retrieval
#   tool call and ends the chat once Critic_Agent approves.

retriever_agent = autogen.ConversableAgent(
    name="Retriever_Agent",
    system_message=(
        "You are the Retriever Agent for a TCS Network Operations Knowledge "
        "Assistant. Your ONLY job is to call the retrieve_context_tool "
        "function with a well-formed search query based on the engineer's "
        "question, then hand the returned SOP context to Answer_Agent. Do "
        "not answer the question yourself. Stay in your lane."
    ),
    llm_config= llm_config,
    human_input_mode="NEVER"
)

answer_agent = autogen.ConversableAgent(
    name="Answer_Agent",
    system_message=(
        "You are a Network Operations Knowledge Assistant for TCS's Network "
        "Infrastructure & Security Division. Using ONLY the SOP context "
        "provided by Retriever_Agent (F5 BIG-IP, Cisco Nexus, VLAN/trunk, "
        "BGP troubleshooting docs), answer the engineer's question following "
        "these rules:\n"
        "1. Answer strictly from the provided context. If the context doesn't "
        "contain the answer, say 'I don't have that information in the "
        "knowledge base' -- do not guess.\n"
        "2. Reproduce command syntax (tmsh, CLI) EXACTLY as it appears in the "
        "context -- never paraphrase or 'correct' it.\n"
        "3. Preserve the exact order of any multi-step procedure (e.g. "
        "upgrade Standby before Active).\n"
        "4. Include any Pass Criteria, Expected Result, or verification "
        "table from the context so the engineer can confirm success.\n"
        "5. If a procedure has a rollback or emergency step (e.g. F5 TAC "
        "contact), surface it proactively for failure-related questions.\n"
        "6. Cite which section the answer came from (e.g. 'Per Section 1.5 "
        "-- Rollback Procedure').\n"
        "7. Keep answers concise and operational -- this is for engineers "
        "actively working on infrastructure, not a general explainer.\n"
        "If Critic_Agent requests a revision, revise and resend. Stay in "
        "your lane."
    ),
    llm_config= llm_config,
    human_input_mode="NEVER"
)

critic_agent = autogen.ConversableAgent(
    name="Critic_Agent",
    system_message=(
        "You are the Critic Agent. Review Answer_Agent's latest answer "
        "against the retrieved SOP context and check ALL of the following:\n"
        "1. Every claim is grounded in the retrieved context -- nothing "
        "invented or pulled from general networking knowledge.\n"
        "2. Command syntax is reproduced exactly, not paraphrased.\n"
        "3. Multi-step procedures preserve their original order.\n"
        "4. Pass Criteria / Expected Result / verification tables are "
        "included if present in the context.\n"
        "5. Rollback or emergency steps are surfaced if the question relates "
        "to a failure scenario and the context contains one.\n"
        "6. The answer cites the SOP section it came from.\n"
        "If ALL checks pass, your reply MUST follow this exact two-part format:\n"
        "APPROVED\n"
        "<the full final answer text here, copied from Answer_Agent's message>\n"
        "Do NOT reply with just the word APPROVED on its own -- always include "
        "the complete answer text below it.\n"
        "If any check fails, explain what's missing and ask Answer_Agent to revise."
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)

tool_executor = autogen.UserProxyAgent(
    name="Tool_Executor",
    is_termination_msg=lambda msg: (
        msg.get("name") == "Critic_Agent" and "APPROVED" in msg.get("content", "")),
    code_execution_config=False,
    human_input_mode="NEVER"
)
print("Agents created: Retriever_Agent, Answer_Agent, Critic_Agent, Tool_Executor")

Agents created: Retriever_Agent, Answer_Agent, Critic_Agent, Tool_Executor


In [31]:
# Register the retrieval tool with AutoGen: Retriever_Agent is
# allowed to call it, Tool_Executor is the one that actually runs
# it during the group conversation.

autogen.register_function(
    retrieve_context_tool,
    caller=retriever_agent,
    executor=tool_executor,
    name = "retrieve_context_tool",
    description="Retrieve the most relevant SOP chunks for a given network operations question."
)
print("Tool Registered")

Tool Registered


In [32]:
# Wire the four agents into a round-robin GroupChat (capped at 12
# rounds) and wrap it in a GroupChatManager that drives whose turn
# it is next.

groupchat = autogen.GroupChat(
    agents=[tool_executor, retriever_agent, answer_agent, critic_agent],
    messages=[],
    max_round=12,
    speaker_selection_method="round_robin"
)

group_manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config,
)

print("GroupChat and GroupChatManager ready.")

GroupChat and GroupChatManager ready.


In [33]:
# End-to-end helper: reset chat state, kick off the group
# conversation via Tool_Executor for a given question, then pull
# out Critic_Agent's APPROVED answer (falling back to
# Answer_Agent's last draft if nothing was approved). Runs one
# sample question as a smoke test.

def ask(question):
    groupchat.messages = []  # reset chat state between questions

    chat_result = tool_executor.initiate_chat(
        group_manager,
        message=question,
    )

    history = chat_result.chat_history  # <-- read from the return value directly

    # Prefer Critic_Agent's approved answer
    for msg in reversed(history):
        if msg.get("name") == "Critic_Agent" and "APPROVED" in (msg.get("content") or ""):
            content = msg.get("content", "").replace("APPROVED", "", 1).strip()
            if content:
                return content

    # Fallback: Answer_Agent's last draft
    for msg in reversed(history):
        if msg.get("name") == "Answer_Agent":
            return msg.get("content", "")

    return "No answer was produced -- check the transcript above for errors."

answer = ask("What's the rollback procedure if the F5 BIG-IP upgrade fails?")
print("\n=== FINAL ANSWER ===")
print(answer)


Tool_Executor (to chat_manager):

What's the rollback procedure if the F5 BIG-IP upgrade fails?

--------------------------------------------------------------------------------

Next speaker: Retriever_Agent

Retriever_Agent (to chat_manager):

***** Suggested tool call (call_bmRScxC61w5Bh7CIA6h4gD0i): retrieve_context_tool *****
Arguments: 
{"query":"rollback procedure for F5 BIG-IP upgrade failure"}
**************************************************************************************

--------------------------------------------------------------------------------

Next speaker: Tool_Executor


>>>>>>>> EXECUTING FUNCTION retrieve_context_tool...
Call ID: call_bmRScxC61w5Bh7CIA6h4gD0i
Input arguments: {'query': 'rollback procedure for F5 BIG-IP upgrade failure'}

>>>>>>>> EXECUTED FUNCTION retrieve_context_tool...
Call ID: call_bmRScxC61w5Bh7CIA6h4gD0i
Input arguments: {'query': 'rollback procedure for F5 BIG-IP upgrade failure'}
Output:
Source : TCS_Network_KB_SOPs.docx
 Content :